# LJ5 — Weighing the experts
### a latency + weight ledger and an honest held-out passport for the models we ship but never trained

*MVM Lab Journal — 5 of a series. Prev: LJ4 (A camera that never lies). Next: LJ6 (Shipping v0.2.0).*

> The logic layers are measured now — pose (LJ1), transitions (LJ2), tracker (LJ3), their
> composition (LJ4). But underneath them sit **seven third-party ONNX experts** the product has
> carried since 2024 without ever measuring their speed or their accuracy on an honest bench. We
> didn't even know our own plumbing wasn't scrambling them: in phase 10.1 an HWC-into-NCHW bug had
> been feeding EmotiEffLib scrambled channels since phase 6 — near-uniform softmax on every face —
> and it was caught **by eye**, not by a number. This journal is the instrument that catches such
> things on the first run. Every accuracy passport opens with a **negative control**: a score at
> chance means OUR wiring is broken, not that the model is bad. Three axes: **how fast, how heavy,
> how accurate** — and one diet: what can the 15.1 results already delete without paying for it.

## 1. Context — five years of unmeasured experts

MVM is an ensemble of specialists. The *logic* around them is now honest end-to-end. The
*specialists themselves* — YuNet (face), SFace (identity), EmotiEffLib (emotion), BlazePose
(person + pose), YOLO26n (objects), Depth-Anything, handpose — were dropped in, wired up, and
trusted. Two things were never established:

1. **Do they fit the budget?** The invariant is `<50 ms` CPU for the human chain, `$0`, no GPU.
   Nobody had a per-agent latency number; "≈12 ms each" was folklore.
2. **Are they as good as claimed — and does our wiring preserve it?** The 10.1 emotion bug proves
   the danger: a model can be fine and the integration can throw it away silently.

YOLO deserves a special note. It is not periphery. It is the scene's **object channel** — the
persistent substrate fusion reasons over. When no human is in frame the scene is still furnished;
the decorations remain and fusion keeps producing structure from them. So YOLO earns a passport as
weighty as the human stack's.

**Hypotheses, with failure criteria named up front (frozen bench-protocol, Act tables):**

| act | hypothesis | failure criterion (declared before the number) |
|---|---|---|
| A | human chain < 50 ms CPU; per-model attributable | over budget → name the dominant agent as a diet/cadence target |
| B1 | emotion balanced-acc ≫ chance, peaked softmax | chance / near-uniform → OUR preprocessing bug (10.1 redux), stop & fix |
| B2 | SFace LFW ≈ published (~0.99) | LFW ≈ 0.5 → broken embedding (the #5 disaster), stop & fix |
| B3 | detectors recall ≫ 0 cross-domain, decode correct | recall ≈ 0 → broken NMS/letterbox, stop & fix |
| C | diet shrinks footprint at no accuracy or latency cost | any diet that costs accuracy OR latency is rejected, measured |

## 2. Setup — everything mounted (no DNS on API-pushed runs)

The ONNX experts and every bench dataset are mounted (RAF-DB, LFW, an OpenImages detect subset).
Latency is **reference-hardware-bound**: the shipping numbers below were measured on an
i7-12700K; the notebook re-runs the per-model decomposition on Kaggle's CPU as an *echo* (a
different chip → different absolute numbers, same shape).

In [ ]:
import os, glob, json, time, sys
import numpy as np

def _find_dir(*names):
    for n in names:
        for root in ("/kaggle/input",):
            hits = [d for d in glob.glob(f"{root}/**/{n}", recursive=True) if os.path.isdir(d)]
            if hits: return hits[0]
    return None

def _find(*names):
    for n in names:
        hits = glob.glob(f"/kaggle/input/**/{n}", recursive=True)
        if hits: return hits[0]
    return None

MODELS_DIR = _find_dir("mvm-onnx-experts", "models") or "."
RAF_DIR    = _find_dir("rafdb", "raf-db-dataset", "DATASET")
LFW_DIR    = _find_dir("lfw_home", "lfw-verification", "lfw_funneled")
def M(fn):  # resolve a model file from the mounted experts dataset
    p = _find(fn); assert p, f"mount mvm-onnx-experts (missing {fn})"; return p
print("models:", MODELS_DIR, "| raf:", RAF_DIR, "| lfw:", LFW_DIR)
print("inputs:", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "no /kaggle/input")

In [ ]:
#!/usr/bin/env python3
"""Frozen bench protocol (phase 15.0.3) — the ONE place every LJ journal computes
metrics and stamps a reproducibility passport, so numbers are comparable across
stages and traceable to exact code+data.

Split policy (FROZEN):
  * clean core (lab+gold5) grouped 5-fold, StratifiedGroupKFold(seed=0), grouping by `grp`.
  * gold is the LOAD-BEARING held-out test (R3): never used to fit anything you report on gold.
  * azimuth bench: mocap P3D re-projected at fixed ANGLES via mocap_to_views.project — TRUE 3D,
    not the circular y-rotation of MediaPipe z.
Reproducibility = fixed seed + fixed data content. `passport()` records the git commit,
working-tree cleanliness, and sha256 of each data file, embedded in every journal's results.

Import, don't fork: from bench import balanced_accuracy, grouped_cv, bootstrap_ci, passport, ANGLES
"""
import os, hashlib, subprocess, numpy as np

CLASSES = ["standing", "sitting", "lying"]
SEED = 0
N_SPLITS = 5
ANGLES = [0, 15, 30, 45, 60, 90]     # azimuth bench, frozen
N_BOOT = 1000                        # grouped bootstrap resamples


# ------------------------------- metrics -----------------------------------
def balanced_accuracy(y, pred, classes=CLASSES):
    """Mean per-class recall (immune to class imbalance) — the headline pose metric."""
    y, pred = np.asarray(y), np.asarray(pred)
    rec = [(pred[y == c] == c).mean() for c in classes if (y == c).any()]
    return float(np.mean(rec))


def per_class_recall(y, pred, classes=CLASSES):
    y, pred = np.asarray(y), np.asarray(pred)
    return {c: (float((pred[y == c] == c).mean()) if (y == c).any() else None) for c in classes}


def confusion(y, pred, classes=CLASSES):
    y, pred = np.asarray(y), np.asarray(pred)
    idx = {c: i for i, c in enumerate(classes)}
    M = np.zeros((len(classes), len(classes)), int)
    for t, p in zip(y, pred):
        if t in idx and p in idx:
            M[idx[t], idx[p]] += 1
    return M


# --------------------------- grouped evaluation ----------------------------
def grouped_cv(X, y, groups, fit_predict, n_splits=N_SPLITS, seed=SEED):
    """StratifiedGroupKFold OOF. fit_predict(Xtr,ytr,Xte)->pred_te. Returns (per_fold_bal, oof_pred)."""
    from sklearn.model_selection import StratifiedGroupKFold
    y = np.asarray(y)
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.empty(len(y), dtype=object)
    per_fold = []
    for tr, te in sgkf.split(X, y, groups=groups):
        pred = np.asarray(fit_predict(X[tr], y[tr], X[te]))
        oof[te] = pred
        per_fold.append(balanced_accuracy(y[te], pred))
    return per_fold, oof


def bootstrap_ci(y, pred, groups, metric=balanced_accuracy, n=N_BOOT, seed=SEED, alpha=0.05):
    """Grouped bootstrap CI (resample GROUPS w/ replacement — matches NB3b). Returns (point, lo, hi)."""
    y, pred, groups = np.asarray(y), np.asarray(pred), np.asarray(groups)
    uniq = np.unique(groups)
    by = {g: np.where(groups == g)[0] for g in uniq}
    rng = np.random.default_rng(seed)
    stats = []
    for _ in range(n):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([by[g] for g in pick])
        stats.append(metric(y[idx], pred[idx]))
    lo, hi = np.percentile(stats, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(metric(y, pred)), float(lo), float(hi)


def azimuth_curve(P3D, y, fit_predict_train, feat_fn, angles=ANGLES):
    """Honest azimuth bench: model fit on training features, tested on mocap re-projected at each
    angle. fit_predict_train is a fitted callable f(feature_matrix)->pred. Returns {angle: bal_acc}."""
    from mocap_to_views import project
    out = {}
    for a in angles:
        F = np.array([feat_fn(project(p, a)) for p in P3D])
        out[a] = balanced_accuracy(y, fit_predict_train(F))
    return out


# ------------------------------- passport ----------------------------------
def _sha256(path, cap=None):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while (chunk := f.read(1 << 20)):
            h.update(chunk)
    return h.hexdigest()[:16]


def passport(data_files=(), root=None):
    """Reproducibility stamp for a journal run: git commit + dirty flag + data sha256s.
    Embed the returned dict in results/metrics.json under the journal's key."""
    root = root or os.path.dirname(os.path.abspath(__file__))
    def git(*a):
        try:
            return subprocess.check_output(["git", "-C", root, *a], stderr=subprocess.DEVNULL).decode().strip()
        except Exception:
            return "?"
    dirty = bool(git("status", "--porcelain"))
    data = {}
    for f in data_files:
        p = f if os.path.isabs(f) else os.path.join(root, f)
        data[os.path.basename(f)] = _sha256(p) if os.path.exists(p) else "MISSING"
    return {"code_commit": git("rev-parse", "--short", "HEAD"),
            "working_tree": "dirty" if dirty else "clean", "data_sha256": data}

## 3. Act A — the latency & weight ledger

Per-model onnxruntime CPU inference (median + p95), and exact file bytes. The human chain runs
its five required experts sequentially per person; the cadenced ones (depth, YOLO) live on their
own threads and are outside the `<50 ms` budget by design (HOL-blocking was removed in phase 8).

**Reference-hardware shipping numbers (i7-12700K, onnxruntime 1.27.0 CPU, 200 timed):**

| agent | role | weight | lat (all cores) | p95 | lat (1 thread) |
|---|---|---:|---:|---:|---:|
| YuNet (face) | chain | 0.23 MiB | 1.88 ms | 2.06 | 7.29 |
| **SFace (identity)** | chain | **36.9 MiB** | **5.20 ms** | 5.41 | 17.52 |
| EmotiEffLib (emotion) | chain | 15.3 MiB | 4.56 ms | 4.87 | 11.44 |
| BlazePose (person det) | chain | 11.4 MiB | 2.19 ms | 2.33 | 8.33 |
| BlazePose (pose) | chain | 5.3 MiB | 2.87 ms | 3.19 | 7.44 |
| YOLO26n (objects) | cadenced | 9.5 MiB | 12.50 ms | 13.22 | 52.72 |
| YOLO-seg (masks) | cadenced | 13.2 MiB | 21.26 ms | 22.24 | 100.0 |
| Depth-Anything | cadenced | 26.0 MiB | 231.8 ms | 240.8 | 939.1 |
| handpose | optional | 3.9 MiB | 0.80 ms | 0.85 | 3.14 |

**Human chain = 16.7 ms all-cores → gate `<50 ms` PASS (3x margin)** (1-thread floor 52.0 ms,
same dominant SFace). Total ONNX = 121.7 MiB. Dominant by latency *and* weight: **SFace (30% of
all ONNX)** — the diet target Act C interrogates. The Kaggle-CPU echo below reproduces the shape.

In [ ]:
#!/usr/bin/env python3
"""LJ5 Act A — latency & weight ledger for the external ONNX experts (phase 15.5).

Measures, on the reference hardware, each production ONNX expert's:
  * weight   — exact file bytes (matches mvm-cli/src/manifest.rs);
  * latency  — per-model onnxruntime CPU inference, median + p95 over timed runs,
               at two thread settings that BRACKET real deploys:
                 - "all"  = intra_op_num_threads=0 (all cores; single-model peak on this box)
                 - "one"  = intra_op_num_threads=1 (portable floor; no multicore advantage)

Why per-model onnxruntime and not the Rust hot-path: the shipped experts are these exact
ONNX bytes run by the same engine (ort crate ≙ onnxruntime). Per-model isolates the
DECOMPOSITION (which model costs what) reproducibly. The full human-chain wall-clock
(incl. preprocessing/alignment/feature-calc + concurrent depth/YOLO contention) is a
live-camera number the owner validates — cited, not fabricated here. Sum-of-required-models
is the CPU *compute budget* of the human chain (preprocessing is cheap arithmetic vs inference).

Passport: CPU model + thread config are stamped alongside the numbers (latency is
reference-hardware-bound by nature). See research/LJ5-DESIGN.md, Act A.

Usage: venv/bin/python spike_pose/lj5_latency.py [--warmup 30] [--timed 200] [--json out.json]
"""
import os, sys, json, time, argparse, platform, subprocess
import numpy as np


# Human chain (manifest `required=true`): runs sequentially inside the Human agent per person.
# Cadenced/optional experts run on their own threads/cadence and are NOT in the <50ms budget.
CHAIN = {"face_detection_yunet_2023mar.onnx", "face_recognition_sface_2021dec.onnx",
         "enet_b0_8_best_vgaf.onnx", "person_detection_mediapipe_2023mar.onnx",
         "pose_estimation_mediapipe_2023mar.onnx"}
# Human-readable agent labels + role, in chain order then cadenced.
ORDER = [
    ("face_detection_yunet_2023mar.onnx",      "YuNet (face detect)",      "chain"),
    ("face_recognition_sface_2021dec.onnx",    "SFace (identity)",         "chain"),
    ("enet_b0_8_best_vgaf.onnx",               "EmotiEffLib (emotion)",    "chain"),
    ("person_detection_mediapipe_2023mar.onnx","BlazePose (person det)",   "chain"),
    ("pose_estimation_mediapipe_2023mar.onnx", "BlazePose (pose)",         "chain"),
    ("yolo_object_detect.onnx",                "YOLO26n (objects)",        "cadenced"),
    ("yolo_seg.onnx",                          "YOLO-seg (masks)",         "cadenced"),
    ("depth_anything_v2_small.onnx",           "Depth-Anything (depth)",   "cadenced"),
    ("handpose_estimation_mediapipe_2023feb.onnx","MediaPipe (handpose)",  "optional"),
]
# Dynamic-dim resolution (documented): depth is patch-14, canonical 518 (mvm-depth lib.rs:90).
DYN = {"batch_size": 1, "batch": 1, "height": 518, "width": 518}


def resolve_shape(shape):
    return [DYN.get(d, d) if not isinstance(d, int) else d for d in shape]


def bench_model(path, warmup, timed, threads):
    import onnxruntime as ort
    so = ort.SessionOptions()
    so.intra_op_num_threads = threads          # 0 = all cores
    so.inter_op_num_threads = threads
    so.log_severity_level = 3                   # silence initializer-in-inputs warnings
    sess = ort.InferenceSession(path, sess_options=so, providers=["CPUExecutionProvider"])
    feeds = {}
    for i in sess.get_inputs():
        shp = resolve_shape(i.shape)
        feeds[i.name] = np.random.rand(*shp).astype(np.float32)
    for _ in range(warmup):
        sess.run(None, feeds)
    ts = []
    for _ in range(timed):
        t0 = time.perf_counter()
        sess.run(None, feeds)
        ts.append((time.perf_counter() - t0) * 1e3)   # ms
    ts = np.array(ts)
    in_shape = resolve_shape(sess.get_inputs()[0].shape)
    return {"median_ms": float(np.median(ts)), "p95_ms": float(np.percentile(ts, 95)),
            "min_ms": float(ts.min()), "input_shape": in_shape}


def passport():
    def git(*a):
        try:
            return subprocess.check_output(["git", "-C", ROOT, *a],
                                           stderr=subprocess.DEVNULL).decode().strip()
        except Exception:
            return "?"
    return {"code_commit": git("rev-parse", "--short", "HEAD"),
            "working_tree": "dirty" if git("status", "--porcelain") else "clean",
            "cpu": platform.processor() or _cpu_name(), "logical_cores": os.cpu_count(),
            "onnxruntime": _ort_ver(), "provider": "CPUExecutionProvider"}


def _cpu_name():
    try:
        for ln in open("/proc/cpuinfo"):
            if ln.startswith("model name"):
                return ln.split(":", 1)[1].strip()
    except Exception:
        return "?"


def _ort_ver():
    import onnxruntime as ort
    return ort.__version__

In [ ]:
# Kaggle-CPU ECHO of the per-model decomposition (chain experts). Absolute ms differ from the
# reference i7-12700K; the point is the shape + that nothing is pathologically slow here.
import onnxruntime as ort
for fn, label, role in ORDER[:5]:
    r = bench_model(M(fn), 20, 100, 0)
    print(f"{label:26s} {os.path.getsize(M(fn))/1024/1024:6.2f} MiB   {r['median_ms']:.2f} ms (Kaggle CPU)")

## 4. Act B1 — emotion, on a held-out it never saw

EmotiEffLib `enet_b0_8_best_vgaf` was trained on AffectNet+VGAF, so **RAF-DB is a clean held-out**.
Preprocessing is bit-faithful to prod (`mvm-emotion::preprocess_chw`: NEAREST 224, ImageNet norm,
planar CHW, RGB, softmax over 8). RAF-DB ships pre-aligned crops → this passports the classifier
head (prod adds YuNet+Umeyama alignment upstream). **Negative control first.**

In [ ]:
#!/usr/bin/env python3
"""LJ5 Act B1 — emotion passport: EmotiEffLib B0 on RAF-DB held-out (phase 15.5).

Honest held-out accuracy for the emotion expert we ship but never trained. The model
(`enet_b0_8_best_vgaf.onnx`) was trained on AffectNet+VGAF → RAF-DB is a clean held-out
(no train contamination). This is also the regression guard our phase-10.1 NCHW bug would
have tripped: if our wiring reproduces near-uniform softmax / chance balanced-acc, that's a
PREPROCESSING BUG, not a bad model — so the negative control runs first.

Preprocessing is BIT-FAITHFUL to prod (`mvm-emotion/src/lib.rs::preprocess_chw`):
  resize 224² NEAREST, RGB, (px/255 − ImageNet_mean)/ImageNet_std, planar CHW, softmax over 8.
RAF-DB ships pre-ALIGNED 100² crops → we skip prod's YuNet+Umeyama align stage and feed the
crop directly: this passports the CLASSIFIER HEAD (design caveat; prod adds alignment upstream).

RAF-DB label(1..7) → EmotiEffLib 8-class output idx (Contempt idx 1 has no RAF GT → argmax-over-7
for the headline; free argmax-8 Contempt-rate reported separately as an honesty sink).

Usage: venv/bin/python spike_pose/lj5_emotion.py [--data /home/dawoh/datasets/rafdb] [--json out.json]
"""
import os, sys, json, glob, argparse
import numpy as np
from PIL import Image


MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)

# EmotiEffLib output order (mvm-core/src/emotion.rs EMOTION_LABELS).
MODEL_LABELS = ["Anger", "Contempt", "Disgust", "Fear", "Happiness", "Neutral", "Sadness", "Surprise"]
# RAF-DB basic label (1..7) → canonical emotion name.
RAF_NAME = {1: "Surprise", 2: "Fear", 3: "Disgust", 4: "Happiness", 5: "Sadness", 6: "Anger", 7: "Neutral"}
# The 7 RAF classes in a fixed reporting order (Contempt excluded — no RAF GT).
CLASSES7 = ["Anger", "Disgust", "Fear", "Happiness", "Neutral", "Sadness", "Surprise"]
KEEP7 = [MODEL_LABELS.index(c) for c in CLASSES7]   # model indices with RAF correspondence
CONTEMPT = MODEL_LABELS.index("Contempt")           # = 1


def preprocess_batch(paths):
    """Bit-faithful to mvm-emotion::preprocess_chw — NEAREST 224, ImageNet norm, planar CHW, RGB."""
    out = np.empty((len(paths), 3, 224, 224), np.float32)
    for i, p in enumerate(paths):
        im = Image.open(p).convert("RGB").resize((224, 224), Image.NEAREST)
        a = np.asarray(im, np.float32) / 255.0          # H,W,3 RGB
        a = (a - MEAN) / STD
        out[i] = np.transpose(a, (2, 0, 1))             # → 3,H,W (planar CHW)
    return out


def load_test(data_root, split="test"):
    paths, y = [], []
    for raf_label, name in RAF_NAME.items():
        for p in glob.glob(os.path.join(data_root, "DATASET", split, str(raf_label), "*")):
            paths.append(p); y.append(name)
    return paths, np.array(y)

In [ ]:
from PIL import Image
import onnxruntime as ort
raf = _find_dir("DATASET") or RAF_DIR
data_root = os.path.dirname(raf) if raf and os.path.basename(raf) == "DATASET" else raf
so = ort.SessionOptions(); so.log_severity_level = 3
sess = ort.InferenceSession(M("enet_b0_8_best_vgaf.onnx"), sess_options=so, providers=["CPUExecutionProvider"])
iname = sess.get_inputs()[0].name
paths, y = load_test(data_root, "test")
probs = np.empty((len(paths), 8), np.float32)
for s in range(0, len(paths), 128):
    ch = paths[s:s+128]; lo = sess.run(None, {iname: preprocess_batch(ch)})[0]
    e = np.exp(lo - lo.max(1, keepdims=True)); probs[s:s+len(ch)] = e / e.sum(1, keepdims=True)
maxp = probs.max(1); ent = (-(probs*np.log2(np.clip(probs,1e-12,1))).sum(1)/3.0)
print(f"[negative control] max-prob {maxp.mean():.3f} (chance .125) | entropy {ent.mean():.3f} (uniform 1) "
      f"-> {'HEALTHY' if maxp.mean()>.25 and ent.mean()<.9 else 'COLLAPSED (bug!)'}")
pred7 = np.array([MODEL_LABELS[KEEP7[i]] for i in probs[:, KEEP7].argmax(1)])
bal = balanced_accuracy(y, pred7, classes=CLASSES7)
print(f"balanced-acc (7-cls) = {bal:.3f} | overall = {(pred7==y).mean():.3f} (chance {1/7:.3f})")
for c in CLASSES7:
    print(f"  {c:10s} recall {per_class_recall(y, pred7, CLASSES7)[c]:.3f}  (n={int((y==c).sum())})")

**Result — hypothesis confirmed.** Negative control HEALTHY (mean max-prob 0.720 vs chance 0.125;
entropy 0.365). **balanced-acc 0.665 [0.643, 0.688], overall 0.726** (chance 0.143) — the crate's
`0.73` claim is not inflated. Strong: Happiness .84 / Sadness .85 / Anger .82 / Fear .74; weak:
Surprise .37 (→Fear) and Disgust .36 (→Anger) — the classic FER confusions, cross-domain, not
wiring. The 10.1 bug does **not** reproduce: our preprocessing is correct. Verdict: **keep.**

## 5. Act B2 — identity, and a 19.5-point finding

SFace on LFW (10-fold, 6000 pairs), preprocessing bit-faithful to `mvm-identity` (planar CHW, BGR,
raw [0,255], L2). Two paths — because prod does **not** align the face before SFace
(`pipeline.rs:470` feeds the raw YuNet bbox crop, unlike emotion which aligns). So we measure both
the shipped path and the reference-aligned path; the delta is a `+0`-weight fix candidate.

In [ ]:
#!/usr/bin/env python3
"""LJ5 Act B2 — identity passport: SFace on LFW verification (phase 15.5).

Honest verification accuracy for the identity expert we ship. SFace (Apache) is the priciest
in-chain model (36.9 MiB, 5.2 ms) → its accuracy headroom decides the Act-C requant verdict.

TWO paths measured, because prod DOES NOT align the face before SFace (pipeline.rs:470 feeds the
raw YuNet bbox crop straight to embed(), unlike emotion which aligns via align_face). So:
  * PROD (unaligned): YuNet bbox → resize 112 → SFace   — what ships today
  * ALIGNED         : YuNet 5-landmark → our Umeyama warp to 112 → SFace   — the 15.6 fix candidate
The delta = the measured cost of not aligning identity (a +0-weight fix, reuses align_face).

SFace preprocessing is BIT-FAITHFUL to prod (`mvm-identity/src/lib.rs::preprocess`):
  planar CHW, **BGR**, **raw [0,255]** (model bakes in (x−127.5)/128), L2-normalise output, cosine match.
Resize is bilinear (≈ prod Triangle). Detection = cv2.FaceDetectorYN on the SAME YuNet ONNX;
its 5 landmarks share order with our ArcFace template (image-left-eye first). Negative control:
LFW acc ≈ 0.5 = broken embedding (the #5 disaster), not a weak model.

Usage: venv/bin/python spike_pose/lj5_identity.py [--lfw .../lfw_home] [--json out.json]
"""
import os, sys, json, argparse
import numpy as np, cv2
from PIL import Image


# ArcFace 5-point template on 112² (mvm-emotion/src/lib.rs ALIGN_TEMPLATE_112), image coords.
TEMPLATE = np.array([[38.2946, 51.6963], [73.5318, 51.5014], [56.0252, 71.7366],
                     [41.5493, 92.3655], [70.7299, 92.2041]], np.float32)


def similarity_matrix(src, dst):
    """Closed-form Umeyama (uniform scale+rot+trans, no reflection) src→dst, as mvm-emotion.
    Returns 2×3 cv2 affine [[a,-b,tx],[b,a,ty]] or None if degenerate."""
    s, d = src.mean(0), dst.mean(0)
    u, v = src - s, dst - d
    num_re = (u[:, 0] * v[:, 0] + u[:, 1] * v[:, 1]).sum()
    num_im = (u[:, 0] * v[:, 1] - u[:, 1] * v[:, 0]).sum()
    den = (u * u).sum()
    if den < 1e-9:
        return None
    a, b = num_re / den, num_im / den
    tx = d[0] - a * s[0] + b * s[1]
    ty = d[1] - b * s[0] - a * s[1]
    return np.array([[a, -b, tx], [b, a, ty]], np.float32)


def sface_embed(sess, crop_bgr112):
    """Bit-faithful SFace preprocess: planar CHW, BGR, raw [0,255]; L2-normalise output."""
    x = np.transpose(crop_bgr112.astype(np.float32), (2, 0, 1))[None]  # 1,3,112,112 (already BGR)
    emb = sess.run(None, {"data": x})[0][0]
    n = np.linalg.norm(emb)
    return emb / n if n > 1e-8 else emb


def make_crops(img_bgr, det):
    """Return (unaligned112, aligned112) BGR crops from a YuNet detection row, or center fallback."""
    h, w = img_bgr.shape[:2]
    if det is None:
        c = img_bgr[h // 5:h * 4 // 5, w // 5:w * 4 // 5]          # center fallback
        un = cv2.resize(c, (112, 112), interpolation=cv2.INTER_LINEAR)
        return un, un
    x, y, bw, bh = det[:4]
    x0, y0 = max(0, int(x)), max(0, int(y))
    x1, y1 = min(w, int(x + bw)), min(h, int(y + bh))
    crop = img_bgr[y0:y1, x0:x1]
    un = cv2.resize(crop, (112, 112), interpolation=cv2.INTER_LINEAR) if crop.size else \
        cv2.resize(img_bgr, (112, 112))
    lm = det[4:14].reshape(5, 2).astype(np.float32)
    M = similarity_matrix(lm, TEMPLATE)
    al = cv2.warpAffine(img_bgr, M, (112, 112), flags=cv2.INTER_LINEAR) if M is not None else un
    return un, al


def parse_pairs(pairs_txt, root):
    """Standard LFW pairs.txt → list of (pathA, pathB, label, fold). 10 folds × (300 same+300 diff)."""
    def p(name, n):
        return os.path.join(root, "lfw_funneled", name, f"{name}_{int(n):04d}.jpg")
    lines = open(pairs_txt).read().splitlines()
    nfold, nper = map(int, lines[0].split())
    out, i = [], 1
    for f in range(nfold):
        for _ in range(nper):                       # matched
            a, n1, n2 = lines[i].split(); i += 1
            out.append((p(a, n1), p(a, n2), 1, f))
        for _ in range(nper):                       # mismatched
            a, n1, b, n2 = lines[i].split(); i += 1
            out.append((p(a, n1), p(b, n2), 0, f))
    return out


def tar_at_far(cos, label, far_target=1e-3):
    same, diff = cos[label == 1], cos[label == 0]
    thr = np.quantile(diff, 1 - far_target)          # threshold giving FAR = far_target
    return float((same > thr).mean()), float(thr)


def tenfold_acc(cos, label, fold):
    """Canonical LFW: threshold from the other 9 folds (max train acc), applied to the held-out fold."""
    accs = []
    grid = np.linspace(-1, 1, 4001)
    for f in range(10):
        tr, te = fold != f, fold == f
        best_t = grid[np.argmax([((cos[tr] > t) == label[tr]).mean() for t in grid])]
        accs.append(((cos[te] > best_t) == label[te]).mean())
    return float(np.mean(accs)), float(np.std(accs))

In [ ]:
import cv2, onnxruntime as ort
so = ort.SessionOptions(); so.log_severity_level = 3
sess = ort.InferenceSession(M("face_recognition_sface_2021dec.onnx"), sess_options=so, providers=["CPUExecutionProvider"])
yunet = cv2.FaceDetectorYN.create(M("face_detection_yunet_2023mar.onnx"), "", (250, 250), 0.6, 0.3, 50)
lfw = _find_dir("lfw_home") or LFW_DIR
pairs = parse_pairs(os.path.join(lfw, "pairs.txt"), lfw)
uniq = sorted({p for pr in pairs for p in pr[:2]})
emb_un, emb_al = {}, {}
for path in uniq:
    img = cv2.imread(path); h, w = img.shape[:2]; yunet.setInputSize((w, h))
    _, faces = yunet.detect(img)
    det = faces[np.argmax(faces[:, -1])] if faces is not None and len(faces) else None
    un, al = make_crops(img, det)
    emb_un[path] = sface_embed(sess, un); emb_al[path] = sface_embed(sess, al)
label = np.array([pr[2] for pr in pairs]); fold = np.array([pr[3] for pr in pairs])
for nm, E in [("PROD (unaligned)", emb_un), ("ALIGNED", emb_al)]:
    cos = np.array([float(E[a] @ E[b]) for a, b, _, _ in pairs])
    acc, sd = tenfold_acc(cos, label, fold); tar, _ = tar_at_far(cos, label)
    print(f"[{nm:16s}] 10-fold acc = {acc:.4f} ± {sd:.4f}   TAR@FAR=1e-3 = {tar:.3f}")

**Result — hypothesis confirmed, with a finding.** The **aligned** path hits **0.9783** = the
published SFace-LFW → the model and our SFace wiring are correct (the negative control is taken
hard; the #5 disaster does not reproduce). But the **shipped, unaligned** path is only **0.7833** —
prod feeds SFace a raw bbox crop and leaves **19.5 points** of verification accuracy on the table
(TAR@FAR=1e-3 collapses 0.949 → 0.173, ~5.5x fewer true accepts at fixed false-accept). The fix is
one line — wrap the crop in `align_face(lm5, 112)`, exactly what emotion already does — **+0 weight,
+0 latency, +19.5 points.** (LFW is funneled, so this headroom is a *lower* bound on real faces.)
This is the star of the journal: the same "measure, don't assume" that caught 10.1, now worth 19.5
points for free.

## 6. Act B3 — the detectors, cross-domain on OpenImages

Recall@IoU0.5 on non-group OI boxes (recall is robust to OI's federated labels; precision is a
lower bound). Cross-domain: YuNet trained on WIDER FACE, YOLO26n on COCO — OI is neither.
Preprocessing bit-faithful to `mvm-scene` (letterbox 640, grey 114, RGB /255; decode xyxy→norm).

**Measured (OpenImages validation):**

- **YuNet (face, in-chain): recall 0.805 prominent / 0.772 all** (543 / 758 GT) — strong, decode
  correct.
- **YOLO26n (person): 0.653 prominent** — nano, cross-domain; prod's person path is BlazePose, not
  YOLO (this is an object-detector proxy).
- **YOLO26n OBJECT passport (18 interaction classes — the scene substrate fusion consumes):**
  MACRO 0.520 / MICRO 0.460 prominent. The **fusion-critical operated objects are strong (0.6–0.8):**
  Mobile phone .81, Remote .78, Laptop .74, Mouse .73, TV .69, Keyboard .60, Clock .68, Vase .67,
  Bowl .66, Wine glass .67. The weak tail is mostly **cross-taxonomy**, not model failure: Book .07
  (OI labels shelf spines; COCO learned held/open books), Table .22 (COCO dining-table ≠ OI Table),
  bags .09–.13. The decode is correct (recall 0.4–0.8, not 0). Verdict: **keep** — YOLO does its
  fusion job (interaction objects) well; the tail is taxonomy + genuine nano limits, a weight↔accuracy
  trade for a future phase, not a wiring bug.

The code (`lj5_detect.py`, `lj5_objects.py`) is in the repo; it runs live over a mounted OI subset.

## 7. Act C — the diet, decided (and one measured negative)

SFace is the heaviest in-chain model (36.9 MiB, 5.2 ms) → the obvious requant target. INT8 dynamic
quantization: **weight 36.9 → 9.4 MiB (−74%), accuracy LOSSLESS** (LFW aligned 0.9790 vs 0.9783,
within CI). It looks free. It is not:

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType
import onnxruntime as ort, numpy as np, time
src_m = M("face_recognition_sface_2021dec.onnx"); dst_m = "/kaggle/working/sface_int8.onnx"
quantize_dynamic(src_m, dst_m, weight_type=QuantType.QInt8)
def lat(p, thr=0):
    so = ort.SessionOptions(); so.intra_op_num_threads = thr; so.log_severity_level = 3
    s = ort.InferenceSession(p, sess_options=so, providers=["CPUExecutionProvider"])
    x = np.random.rand(1,3,112,112).astype(np.float32)
    for _ in range(20): s.run(None, {"data": x})
    t = [time.perf_counter() for _ in range(1)]
    ts = []
    for _ in range(100):
        a = time.perf_counter(); s.run(None, {"data": x}); ts.append((time.perf_counter()-a)*1e3)
    return np.median(ts)
print(f"fp32 {os.path.getsize(src_m)/1024/1024:.1f} MiB  {lat(src_m):.2f} ms")
print(f"int8 {os.path.getsize(dst_m)/1024/1024:.1f} MiB  {lat(dst_m):.2f} ms  <- ~6x SLOWER on CPU")

**On the reference hardware INT8 is ~6x slower (5.10 → 31.63 ms all-cores)** — the classic
dynamic-quant-on-CNN pitfall (DynamicQuantizeLinear + INT8 matmul lose to AVX-tuned fp32 conv).
It would push the human chain 16.7 → ~43 ms and violate the latency gate. **Verdict: REJECT** —
a measured negative. (Static quantization with calibration is a future option, uncertain on CNN CPU.)

The real diet was already banked in 15.1 at the **arithmetic layer**, where footprint is ours to cut:

| layer | v0.1.0 | v0.2.0 candidate | Δ |
|---|---:|---:|---:|
| ONNX experts (9) | 121.7 MiB | 121.7 MiB | 0 (requant rejected) |
| arithmetic (embedded) | 1068 KB (forest 873 + VAE 195) | **222 KB** (forest 220 + Mahalanobis 1.8) | **−79%** |

Total footprint **shrinks**; the human chain stays 16.7 ms. **Weight/latency gate: PASS.**

## 8. Verdict

| hypothesis | outcome |
|---|---|
| A: chain < 50 ms, attributable | **confirmed** — 16.7 ms all-cores; dominant = SFace (named) |
| B1: emotion ≫ chance, peaked | **confirmed** — 0.665 balanced-acc, wiring HEALTHY (10.1 not reproduced) |
| B2: SFace ≈ published | **confirmed + finding** — aligned 0.978, but prod ships unaligned 0.783 (**+19.5 pts on the table**) |
| B3: detectors recall ≫ 0, decode ok | **confirmed** — YuNet face 0.805; YOLO interaction objects 0.6–0.8 |
| C: diet at no cost | **confirmed** — arithmetic −79%; SFace INT8 **rejected** (measured +6x latency) |

Every negative control was taken: **our wiring breaks no model.** The load-bearing result is that
the biggest quality win in the whole passport costs nothing — **align the identity crop.**

## 9. What it changes — the bridge to LJ6 (shipping v0.2.0)

Into the 15.6 port package, each licensed by a number here:

1. **Align the identity crop** — wrap the SFace input in `align_face(lm5, 112)` (reuse emotion's).
   +0 weight, +0 latency, **+19.5 points** LFW verification. The single best value in the passport.
2. **Diet** — drop the VAE (Mahalanobis won, 15.1), ship the 220 KB forest: arithmetic −79%.
3. **Keep every ONNX expert as-is** — none is both a weight problem and safe to shrink; INT8 SFace
   is a documented reject.
4. **Quality passport in README/AGENT.md** — the numbers above, each traceable to this journal.

What this journal honestly does not do: retrain anything (out of scope); measure the full live
contended chain (per-model isolated + the owner-validated ~16 fps engine is the ceiling); or fix
the cross-taxonomy floor on YOLO's furniture/book classes (a weight↔accuracy trade for later).

Next, **LJ6 — Shipping v0.2.0: the scorecard** — the Rust port of every winning decision from
LJ1–LJ5, and the before/after across all three axes.

In [ ]:
# ---- passport: bind the numbers to code + data (+ commit when run from the repo) ----
import hashlib, subprocess
def sha16(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for ch in iter(lambda: f.read(1 << 20), b""): h.update(ch)
    return h.hexdigest()[:16]
try:
    commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL).decode().strip()
except Exception:
    commit = "(kaggle run — no git)"
passport = {"code_commit": commit,
            "sface": sha16(M("face_recognition_sface_2021dec.onnx")),
            "emotion": sha16(M("enet_b0_8_best_vgaf.onnx")),
            "cpu_note": "latency numbers in the tables are reference-hardware (i7-12700K); "
                        "Kaggle echoes above are a different chip"}
print("passport:", passport)